# Deep Q-Network Assigment


# Background

The purpose of this assignment is to implement a Deep Q-Network to solve a simple gridworld environment. You will be implementing a neural network function approximator, Experience Replay and Fixed Q-value Targets. This assignment builds on top of the Tabular Q-Learning assignment. We assume that you have succesfully managed to implement the tabular Q-learning agent. We will now discuss the additional components that will be added. We will be using PyTorch as neural network library.

In order to implement a Deep Q-Network, there are a couple of steps to understand, including:
1. The Q-Learning training algorithm
2. The gridworld environment
3. Preproces observations received from the environment to obtain the current state
4. Create a Neural Network architecture for the policy network (to estimate policy) and target network (to calulate TD-targets) 
5. Create Experience Replay Memory to store generated experience
6. Define a loss function
7. Define and configure an optimiser
8. Sample a mini-batch of experience and optimise the policy network model parameters
9. Periodically update the Fixed Q-value target network
10. Evaluate the performance of the trained policy network

# Training a Deep Q-Network algorithm


Our algorithm is summarized below:
<br>
* Initialize the weights of the policy network and target networks
* Initialize the environment
* Set the decay rate (that will use to calculate the $\epsilon_{\textrm{threshold}}$)
* Set fixed Q-value target update threshold
* Set total training steps to 0
* Create replay memory $\mathcal{D}$
<br><br>
* **For** episode to max_episode **do** 
    * Set step to 0
    * Make new episode
    * Observe the first state $s$
    <br><br>
    * **While** {(not done) and (step < max_steps)} **do**:
        * With $\epsilon$ select a random action $a$, otherwise select $a = \mathrm{argmax}_a Q(s,a)$
        * Increment the total training steps 
        * Execute action $a_t$ in environment, observe reward $r$ and new state $s'$
        * Store transition $<s, a, r, s'>$ in replay memory $\mathcal{D}$
        * **If** size($\mathcal{D}$) >= mini-batch size $N$: 
            * Sample random mini-batch from $\mathcal{D}$: $<s_i, a_i, r_i, s_i'>$ with $i=1\ldots N$
            * Set TD-target $\hat{Q} = r$ if the episode ends at $+1$, otherwise set $\hat{Q} = r + \gamma \max_{a'}{Q(s', a')}$
            * Make a gradient descent step with loss $(\hat{Q} - Q(s, a))^2$
        * **If** total_steps > fixed Q-value target update threshold:
            * Copy parameters from policy network to target network
    * **endwhile**
    <br><br>
* **endfor**

**The skeleton of the training algorithm is shown below. It will not (yet) run correctly as all the libraries have not been imported and the various function that are needed have not been written.**

In [1]:
# Make the gym environment
env = gym.make('MiniGrid-Empty-8x8-v0')

# Use a wrapper so the observation only contains the grid information
env = ImgObsWrapper(env)

episodes = 2               # total number of training episodes
max_steps = env.max_steps  # maximum number of steps allowed before truncating episode
steps_done = 0             # total training steps taken

print('Start training...')
for e in range(episodes):
    
    # reset the environment
    obs, _ = env.reset()

    # extract the current state from the observation
    state = preprocess(obs)
    
    for i in range(0, max_steps):

        # Choose an action
        # Pick a random action
        action = select_action(state)
        a = action.item()
        
        # take action 'a', receive reward 'reward', and observe next state 'obs'
        # 'done' indicate if the termination state was reached
        obs, reward, done, truncated, info = env.step(a)
   
        # extract the next state from the observation
        nextState = preprocess(obs)
        
        # if the episode is finished, the nextState is set to None to indicate that the
        # <s,a,r,s'> transition led to a terminating state
        if (done or truncated):
            nextState = None
        
        # Store the transition <s,a,r,s'> in the replay memory

        # Move to the next state          
        currentState = nextState

        # Perform one step of the optimization (on the policy network) by
        # sample a mini-batch and train the model using the sampled mini-batch
        
        
        # If the target update threshold is reached, update the target network, 
        # copying all weights and biases in the policy network   

        
        # Episode finished when done or truncated is true
        if (done or truncated):
            # Record the reward and total training steps taken
            if (done):
                # if agent reached its goal successfully
                print('Finished episode successfully taking %d steps and receiving reward %f' % (env.step_count, reward))
            else:
                # agent failed to reach its goal successfully 
                print('Truncated episode taking %d steps and receiving reward %f' % (env.step_count, reward))
            break
            
        
print('Done training...')

NameError: name 'gym' is not defined

# Step 1: Import the libraries

In [2]:
# import 'gymnasium' and 'minigrid' for our environment
import gymnasium as gym
import minigrid

# import 'random' to generate random numbers
import random

# import 'numpy' for various mathematical, vector and matrix functions
import numpy as np

from os.path import exists

# import 'Pytorch' for all our neural network needs

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

# if gpu is to be used, otherwise use cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Import 'namedtuple' and 'deque' for Experience Replay Memory
from collections import namedtuple, deque

# Step 2: Create the 'minigrid' environment

## Minigrid Gridworld Environment

The gridworld environment we will be using was written by Maxime Chevalier-Boisvert and can be obtained from Github

https://github.com/Farama-Foundation/gym-minigrid

or it can be installed via pip (`pip install minigrid`) or via conda (`conda install minigrid`)


![Title](figures/empty-env.png)

This environment is an empty room, and the goal of the agent is to reach the green goal square, which provides a sparse reward. A small penalty is subtracted for the number of steps to reach the goal. This environment is useful, with small rooms, to validate that your RL algorithm works correctly, and with large rooms to experiment with sparse rewards and exploration. The random variants of the environment have the agent starting at a random position for each episode, while the regular variants have the agent always starting in the corner opposite to the goal.

### Structure of the world:

- The world is an NxM grid of tiles
- Each tile in the grid world contains zero or one object
- Cells that do not contain an object have the value None
- Each object has an associated discrete color (string)
- Each object has an associated type:
    - 'unseen'        : 0
    - 'empty'         : 1
    - 'wall'          : 2
    - 'floor'         : 3
    - 'door'          : 4
    - 'key'           : 5
    - 'ball'          : 6
    - 'box'           : 7
    - 'goal'          : 8
- The agent can pick up and carry exactly one object (eg: ball or key)
- To open a locked door, the agent has to be carrying a key matching the door's color

The minigrid environment is compatible with OpenAI's gym environments. Any RL algorithm that is written to work with an OpenAI gym environment should also work with the minigrid environment.

The minigrid environment can be imported using the following commands:

In [6]:
# Make the gym environment
env = gym.make('MiniGrid-Empty-8x8-v0')

# If you want to render the environment, you need to add the render_mode='human' parameter
#env = gym.make('MiniGrid-Empty-8x8-v0', render_mode='human')

## Actions in the basic environment:

The environment defines the following basic actions that the agent can perform each step:``

- Turn left
- Turn right
- Move forward
- Pick up an object
- Drop the object being carried
- Toggle (open doors, interact with objects)

We will only be using the first three actions, i.e. "Turn left", "Turn right" and "Move forward".

## Reward Signal

The empty environment we will be using provide sparse reward. Thus the agent only receives a non-zero reward when the goal is successfully achieved. The reward upon success is determined as follows:

$r = 1 - 0.9\times \frac{\textrm{number of steps}}{\textrm{maximum number of steps}}$

The maximum number of steps of the empty environment is $4\times N\times M$ where $N\times M$ is the size of the grid. Eg. for a $8\times 8$ empty grid the maximum number of steps is 256 steps.

### Variables return from step() function

The environment returns five variables from the step() function:

[observation, reward, done, truncated, info] = env.step(...) 

- **observation** - the observation returned by the environment (as detailed above)
- **reward** - the scalar reward associated with taking the action and reaching the next state
- **done** - a boolean variable indicating if the agent reached a terminal state
- **truncated** - a boolean variable indicating if the agent did not manage to reach a terminal state, but the episode ended because the maximum number of steps was reached
- **info** - contains auxiliary diagnostic information


### Episode done
There are two instances when the episode is done:
1. When the agent succesfully reached the goal (receiving a non-zero reward), or
2. When the agent has reached the maximum number of steps of an episode (receiving a reward of zero)

It is important to reset the environment after each episode has ended. This sets the number of steps the agent has taken to zero, and puts the agent back on its starting position. Resetting the environment is also used to obtain the first observation representing the current state $S$.

In [4]:
# reset the environment
obs, info = env.reset()
print(obs)
print(info)

{'image': array([[[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0],
        [2, 5, 0]],

       [[2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]],

       [[2, 5, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]]], dtype=uint8), 'direction': 0, 'mission': 'get

# Step 3: Define the preprocessing functions

## Observation returned by Environment

The default observation that is provided by the environment contains information that is superfluous (such as the 'direction' and the 'mission' - we are only interested in the 'image' data, though the direction data could also be useful). We use a wrapper that is provided by the minigrid environment to only extract the observation representing the partial view that the agents has of the environment.

In [7]:
from minigrid.wrappers import *
env = ImgObsWrapper(env)
env.reset()

(array([[[2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0]],
 
        [[2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0]],
 
        [[2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0],
         [2, 5, 0]],
 
        [[2, 5, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0]],
 
        [[2, 5, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0]],
 
        [[2, 5, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0]],
 
        [[2, 5, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0],
         [1, 0, 0]]], 

Lastly the 'image' observation contains information about each tile around the agent. Each tile is encoded as a 3 dimensional tuple: (OBJECT_IDX, COLOR_IDX, STATE), where OBJECT_TO_IDX and COLOR_TO_IDX mapping can be found in 'minigrid/minigrid.py' and the STATE can be as follows: 

    door STATE -> 0: open, 1: closed, 2: locked

For this task we are only going to be using the information contained in the OBJECT_IDX, therefore we write a small function to extract the OBJECT_IDX information.

In [8]:
# extract the object_idx information as a matrix
def extractObjectInformation(observation):
    (rows, cols, x) = observation.shape
    view = np.zeros((rows, cols))
    for r in range(rows): 
        for c in range(cols): 
            view[r,c] = observation[r,c,0]
    return view

# the following is a more efficient method of extracting the information using numpy slicing and reshaping
def extractObjectInformation2(observation):
    (rows, cols, x) = observation.shape
    tmp = np.reshape(observation,[rows*cols*x,1], 'F')[0:rows*cols]
    return np.reshape(tmp, [rows,cols],'C')
    
obs, _ = env.reset()
print(extractObjectInformation2(obs))

[[2 2 2 2 2 2 2]
 [2 2 2 1 1 1 1]
 [2 2 2 1 1 1 1]
 [2 2 2 1 1 1 1]
 [2 2 2 1 1 1 1]
 [2 2 2 1 1 1 1]
 [2 2 2 1 1 1 1]]


The partial observation provided by the environment is a 7x7 grid. The observation start from the cell furthest from the direction the agent is looking on the left. Indexing is from leftmost row to right most row (row is defined as parallel to direction agent is looking). Within a row, the indexing is from furthers cell to nearest cell. In the example below the whole observation is a 7x7 matrix 'obs'. The 'x' is at index [0][0]. The '>' is at index obs[3][6]. The '+' is at index obs[6][6]. The agent will always be index [3][6]. The location right in front of the agent will always be at index [3][5]. 

**Note that the observation used in this assignment does not contain '>', '+' or 'x' - it is only used for illustration.**

To be used as input to our neural network, the 7x7 matrix 'obs' need to be normalized and flattened. The observation can be normalised by dividing each element by 10 - the maximum object ID in the environment.

In [9]:
# Normalise the input observation so each element is a scalar value between [0,1]
def normalize(observation, max_value):
    return np.array(observation)/max_value

# Flatten the [7,7] observation matrix into a [1,49] tensor
def flatten(observation):
    return torch.from_numpy(np.array(observation).flatten()).float().unsqueeze(0)

# Combine all the preprocessing fuctions into a single function
def preprocess(observation):
    return flatten(normalize(extractObjectInformation2(observation), 10.0))

obs, _ = env.reset()
state = preprocess(obs)
print(state)

tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000, 0.1000,
         0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000, 0.1000, 0.1000, 0.1000,
         0.1000, 0.2000, 0.2000, 0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000,
         0.2000, 0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000,
         0.1000, 0.1000, 0.1000, 0.1000]])


# Step 4: Setup Hyperparameters
In this part we'll set up our different hyperparameters.

- First, you begin by defining the neural networks hyperparameters when you implement the model.
- Then, you'll add the training hyperparameters when you implement the training algorithm.

In [10]:
### MODEL HYPERPARAMETERS 
numActions = 3               # 3 possible actions: left, right, move forward
inputSize = 49               # size of the flattened input state (7x7 matrix of tile IDs)

### TRAINING HYPERPARAMETERS
alpha = 0.0002               # learning_rate
episodes = 5000              # Total episodes for training
batch_size = 128             # Neural network batch size
target_update = 20000        # Number of episodes between updating target network

# Q learning hyperparameters
gamma = 0.90                 # Discounting rate

# Exploration parameters for epsilon greedy strategy
start_epsilon = 1.0          # exploration probability at start
stop_epsilon = 0.01          # minimum exploration probability 
decay_rate = 20000           # exponential decay rate for exploration prob

### MEMORY HYPERPARAMETERS
pretrain_length = batch_size # Number of experiences stored in the Memory when initialized for the first time
memorySize = 500000          # Number of experiences the Memory can keep - 500000

### TESTING HYPERPARAMETERS
# Evaluation hyperparameter
evalEpisodes = 1000          # Number of episodes to be used for evaluation

# Change this to 'False' if you only want to evaluate a previously trained agent
train = True                 # Specify True to train a model, otherwise only evaluate

# Step 5: Define and create our Neural Network model

This is our Deep Q-learning model:

![Title](figures/feedforward.png)

- We take the preprocessed and flattened observation as input state
- We pass it through 2 fully-connected, feed-forward layers
- Each fully-connected, feed-forward layer has a ReLU non-linear activation function
- It outputs a Q value for each action

In [19]:
### Neural network model definition
class DQN(nn.Module):

    def __init__(self, inputSize, numActions, hiddenLayerSize=(512, 256)):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(inputSize, hiddenLayerSize[0])
        self.fc2 = nn.Linear(hiddenLayerSize[0], hiddenLayerSize[1])
        self.fc3 = nn.Linear(hiddenLayerSize[1], numActions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = x.to(device)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
# Instantiate the policy network and the target network

hiddenLayerSize = (128,128)
policy_net = DQN(inputSize, numActions, hiddenLayerSize)
target_net = DQN(inputSize, numActions, hiddenLayerSize)

# Copy the weights of the policy network to the target network
target_net.load_state_dict(policy_net.state_dict())

# We don't want to update the parameters of the target network so we set it to evaluation mode
target_net.eval()

DQN(
  (fc1): Linear(in_features=49, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=3, bias=True)
)

## Saving and loading of networks

The neural network models can be saved and loaded using the builtin Pytorch functions as follows:

In [18]:
filename = 'test.pth'

# Saving a model
torch.save(policy_net, filename)

# Loading a model
loaded_model = torch.load(filename) 

## Performing a forward pass of the policy network using a state as input

A typical computation in DQN is to apply the state as input to the neural network and calculate an estimate of the Q-values for each action: $$\hat{Q}(s,a,\mathbf{w})\quad\forall a\in\mathcal{A}$$

This can be done as follows:

In [19]:
# For this example we first need to get an observation by resetting the environment
obs, _ =  env.reset()

# We then preprocess the observation to obtain our state
state = preprocess(obs)

# Lastly we apply the state as input to our policy network
action_values = policy_net(state)

print('action_values: ',action_values)

# If want want to get the action that has the highest Q-value we use the 'max' function. 
# The result is a tuple where the first element is the value, and the second element is the index
action_values.max(1)
print('\nbest action: ', action_values.max(1))
a = action_values.max(1)[1]
print('\na: ', a)


action_values:  tensor([[ 0.0079, -0.0540,  0.0170]], grad_fn=<AddmmBackward0>)

best action:  torch.return_types.max(
values=tensor([0.0170], grad_fn=<MaxBackward0>),
indices=tensor([2]))

a:  tensor([2])


## $\epsilon$-greedy action selection using the policy network

When the agent initially starts training, it does not know anything about the environment, or which actions will result in a positive reward. The agent therefore initially needs to take random actions in order to _explore_ and learn about the environment. When the agent has learnt more about the environment (by estimating the value-function), the agent can start to _exploit_ the environment, by taking an action that maximimes its expected total reward. One algorithm that is used to control the amount of exploration vs. exploitation is called _Epsilon-Greedy Exploration_.

The main idea is that initially a $\epsilon_{\textrm{threshold}}$ is set to a value $\epsilon_{\textrm{max}}$ where $\epsilon_{\textrm{max}} \leq 1$. The $\epsilon$-value is gradually decreased to a minimum value $\epsilon_{\textrm{min}}$ where $\epsilon_{\textrm{min}}\geq 0$. This can be implemented as an exponential decay function:

$$\epsilon_{\textrm{threshold}} = \epsilon_{\textrm{min}} + (\epsilon_{\textrm{max}}-\epsilon_{\textrm{min}})e^{-\frac{\mbox{steps done}}{\mbox{decay rate}}}$$

At each step, when the agent needs to select an action, it generates a random value between 0 and 1. If the value is smaller than the current $\epsilon_{\textrm{threshold}}$, the agent _explores_ the environment by taking a random action. If the value is bigger than the current $\epsilon_{\textrm{threshold}}$, the agent _exploits_ what is has learnt by taking an action that will maximise its expected reward. This is done by taking the action, which will result in the highest value-function, given the current state of the environment (the state is extracted from the current observation).

This can be implemented as follows:

In [11]:
## Function to e-greedily select next action based on current state
def select_action(state):
    # generate a random number
    sample = random.random()
    
    # calculate the epsilon threshold, based on the epsilon-start value, the epsilon-stop value, 
    # the number of training steps taken and the epsilon decay rate
    # here we are using an exponential decay rate for the epsilon value
    eps_threshold = stop_epsilon+(start_epsilon-stop_epsilon)*math.exp(-1. * steps_done / decay_rate)
    
    # compare the generated random number to the epsilon threshold
    if sample > eps_threshold:
        # act greedily towards the Q-values of our policy network, given the state
        
        # we do not want to gather gradients as we are only generating experience, not training the network
        with torch.no_grad():
            # t.max(1) will return largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1)[1].unsqueeze(0)
    else:
        # select a random action with equal probability
        return torch.tensor([[random.randrange(numActions)]], device=device, dtype=torch.long)

# Step 6: Experience Replay Memory


We’ll be using experience replay memory for training our DQN. It stores the experience that the agent observes, allowing us to reuse this data later. By sampling from it randomly, the experience that build up a batch are decorrelated. It has been shown that this greatly stabilizes and improves the DQN training procedure.

For this, we’re going to need two classses:

Experience - a named tuple representing a single transition in our environment. It essentially maps (state, action) pairs to their (next_state, reward) result, with the state being a preprocessed version of observation obtained from the environment

ReplayMemory - a cyclic buffer of bounded size that holds the experience observed recently. It also implements a .sample() method for selecting a random batch of experience for training. The ReplayMemory uses a deque as internal data structure. A deque (double ended queue) is a data type that **removes the oldest element each time that you add a new element.**

This part was taken from the Pytorch tutorial on Reinforcement Learning (DQN) [https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html]

In [4]:
from collections import namedtuple, deque

Transition = namedtuple('Transition',
                        ('currentState', 'action', 'nextState', 'reward'))

class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)
    
# Instantiate memory
memory = ReplayMemory(memorySize)

The code below shows how a single transition is generated and stored in the Replay Memory

In [13]:
# For this example we first need to get an observation by resetting the environment
obs, _ =  env.reset()
steps_done = 0

# Preprocess the observation to obtain our state
state = preprocess(obs)

# Perform epsilon-greedy action election
action = select_action(state)
a = action.item()

# Perform the action in the environment, received the next state observation and reward
obs, reward, done, truncated, info = env.step(a)
reward = torch.tensor([reward], device = device)

# Preprocess the observation to obtain our next state
nextState = preprocess(obs)

# Store the transition in the Experience Replay Memory
memory.push(state, action, nextState, reward)
print('Number of transition in replay memory: ',len(memory))

# Sample and display a mini-batch of size 1
minibatch = memory.sample(1)
print(minibatch)

Number of transition in replay memory:  2
[Transition(currentState=tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000, 0.1000,
         0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000, 0.1000, 0.1000, 0.1000,
         0.1000, 0.2000, 0.2000, 0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000,
         0.2000, 0.2000, 0.1000, 0.1000, 0.1000, 0.1000, 0.2000, 0.2000, 0.2000,
         0.1000, 0.1000, 0.1000, 0.1000]]), action=tensor([[0]]), nextState=tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000,
         0.1000, 0.1000, 0.1000, 0.1000]]), rew

# Step 7: Set up Tensorboard

For more information about tensorboard, please watch this <a href="https://www.youtube.com/embed/eBbEDRsCmv4">excellent 30min tutorial</a> <br><br>

To launch tensorboard : `tensorboard --logdir=runs`

In [23]:
# Tensorboard writer
writer = SummaryWriter()

# Step 8: Define the loss function

We use the Mean-Squared-Error (MSE) as loss function $\mathcal{L}$. The MSE loss is calculated between the TD-target (calculated using the target model $\hat{Q}(s_i,a',\mathbf{w}^{-})$) and the Q-values predicted by our policy network $\hat{Q}(s'_{i}, a_{i}, \mathbf{w})$.

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\left[R_{i}+\gamma\max_{a'}\hat{Q}(s_i,a',\mathbf{w}^{-}) - \hat{Q}(s'_{i}, a_{i}, \mathbf{w})\right]^{2}$$
where $\left< s_{i}, a_{i}, r_{i}, s'_{i}\right>$ is the $i^{th}$ experience in the mini-batch sampled from the Experience Replay Buffer $\mathcal{D}$

In [14]:
# Compute MSE loss
criterion = nn.MSELoss()

# Step 9: Define and configure the optimiser

We will use the Adam optimiser to train the policy network. The Adam optimiser includes a number of useful gradient descent features including momentum. We specify the learning rate $\alpha$ as one of the parameters of the optimiser.

In [24]:
## Create the optimizer
optimizer = optim.Adam(policy_net.parameters(), lr=alpha)

# Step 10: Train the model using a mini-batch

To train and optimize the model we need to follow the following steps:

 * Sample random mini-batch from $\mathcal{D}$: $<s, a, r, s'>$
 * **For** each transition $<s_{i}, a_{i}, r_{i}, s'_{i}>$ in the mini-batch **do** 
     * Calculate the action-value predicted by the policy network $Q(s_{i},a_{i},\mathbf{w})$
     * Calculate the TD-target estimated by the target network $\hat{Q}(s'_{i}, a'_{i}\mathbf{w}^{-})$:
         * Set TD-target $\hat{Q} = r_{i}$ if the episode ends at $+1$
         * otherwise set $\hat{Q} = r_{i} + \gamma \max_{a'}{Q(s'_{i}, a')}$
 * Calculate the loss using the minimum-squared-error (MSE) criterion 
 $$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\left[R_{i}+\gamma\max_{a'}\hat{Q}(s_i,a',\mathbf{w}^{-}) - \hat{Q}(s'_{i}, a_{i}, \mathbf{w})\right]^{2}$$
 * Make a gradient descent step that will minimise the loss $\mathcal{L}$

The minibatch can be represented as a number of arrays:
$\left(\begin{array}{c}(s_{1})\\(s_{2})\\ \vdots\\(s_{N})\end{array}\right)\left(\begin{array}{c}a_{1}\\a_{2}\\ \vdots\\a_{N}\end{array}\right)\left(\begin{array}{c}r_{1}\\r_{2}\\ \vdots\\r_{N}\end{array}\right)\left(\begin{array}{c}(s'_{1})\\(s'_{2})\\ \vdots\\(s'_{N})\end{array}\right)$

## Sample a mini-batch from the replay memory

In [17]:
# sample a mini-batch of experience from the replay memory
batch_size = 2
experience = memory.sample(batch_size)

# Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
# detailed explanation). This converts batch-array of Experience
# to Experience of batch-arrays.
batch = Transition(*zip(*experience))

## Calculate the action-value predicted by the policy network

This calculation is fairly straightforward. We need to extract the array of 'current_states' (as as Pytorch tensor) and the array of 'actions' (also as Pytorch tensor) from the mini-batch. We then need to perform a forward pass of the policy network using the tensor of 'current states' as input. Pytorch expects the tensor of inputs to a neural network to be of the shape [batch_size, (input_shape)]. For a batch size of 2 and a flattened input vector of size 49, the input tensor needs to be of shape [2, 49]

Our policy network has an ouput layer for size 3 (since our agent can select 3 actions). Performing a forward pass with a batch size of 2 will results in an output tensor of shape [2, 3]. We then need to gather the Q-value associate with each state-action pair $(s_{i}, a_{i})$ which can done using the gather function on the first dimension of the output tensor and specifiying the array of 'actions'. 

In [20]:
# Extract the array of states and array of actions in the mini-batch
state_batch = torch.cat(batch.currentState)
action_batch = torch.cat(batch.action)

# Calculate the action-values for each state in the batch, and 
# then gather the Q-value for the action associated with a specific state
state_action_values = policy_net(state_batch).gather(1, action_batch)
print(state_action_values)

tensor([[ 0.0286],
        [-0.0199]], grad_fn=<GatherBackward0>)


## Calculate the TD-target estimated by the target network

This calculation is of the TD-target is more complicated. First we need to extract the array of 'rewards' and the array of 'next_states' (both as Pytorch tensors) from the mini-batch.

We then need to distinguish between 'next_states' that are final states, and 'next_states' that are non-final states. Final states are indicated by a 'next_state' having a value of 'None'. 

* When the 'next_state' is final the TD-target is just the 'reward'
* When the 'next_state' is non-final the TD-target is the reward plus the discounted, maximum Q-value estimated by the target network for 'next_state'

We can efficiently calculate the TD-targets for the mini-batch by doing the following:
* Initialize the array of 'next_state' values to be an array of zeroes of the same size as the mini-batch
* Calculate 'next_state' values $\max\limits_{a'}Q(s',a')$ but only for the non-final states
* Set the TD-target to be equal to the rewards
* Add the discounted $\max\limits_{a'}Q(s',a')$ to the TD-targets, but only for the non-final states

We will use the concept of masking a tensor to efficiently perform that calculation involving non-final states

In [21]:
# Extract the array of rewards
reward_batch = torch.cat(batch.reward)

# Extract the non-final next states and concatenate them
non_final_next_states = torch.cat([s for s in batch.nextState
                                                if s is not None])

# Initialize the next_state values to be zeroes
next_state_values = torch.zeros(batch_size, device=device)

# Compute a mask of non-final states and concatenate the batch elements
# (a final state would've been the one after which the episode ended)
# This is just a tensor of boolean values, one for each experience in the mini-batch
non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.nextState)), device=device, dtype=torch.bool)

# Calculate the estimated 'next_states' values, but only for the non-final-sates
next_state_values[non_final_mask] = target_net(non_final_next_states).max(1)[0].detach()

# Compute the expected Q values (the TD-target)
TDtargets = (next_state_values * gamma) + reward_batch

# Compute the TDerrors
TDerrors = TDtargets.unsqueeze(1) - state_action_values

print('\nTD-targets: ', TDtargets.unsqueeze(1))
print('\nQ(S,A): ', state_action_values)
print('\nTD-errors: ', TDerrors)


TD-targets:  tensor([[0.0236],
        [0.0280]])

Q(S,A):  tensor([[ 0.0286],
        [-0.0199]], grad_fn=<GatherBackward0>)

TD-errors:  tensor([[-0.0050],
        [ 0.0478]], grad_fn=<SubBackward0>)


## Calculate the loss using the minimum-squared-error (MSE) criterion 
 $$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\left[R_{i}+\gamma\max_{a'}\hat{Q}(s_i,a',\mathbf{w}^{-}) - \hat{Q}(s'_{i}, a_{i}, \mathbf{w})\right]^{2}$$

In [22]:
# Configure the MSELoss
criterion = nn.MSELoss()

# Calculate the loss for the mini-batch. 
# Make sure to resize the TD-targets to be the same shape as the state-action values tensor
loss = criterion(state_action_values, TDtargets.unsqueeze(1))

## Make a gradient descent step that will minimise the loss

In [25]:
# zero the gradients
optimizer.zero_grad()

# calculate the gradients by backpropagation from the loss function
loss.backward()
    
# We clamp the gradients in the policy network to the range [-1,1]
# This also helps keep training stable
for param in policy_net.parameters():
    param.grad.data.clamp_(-1, 1)
    
# Update the network parameters of the policy network using the optimizer
optimizer.step()

## Combine everything in one 'optimize_model' function

We can combine all of the above into one function

In [35]:
## Training of the model
def optimize_model():

    # check if the replay memory has stored enough experience
    if len(memory) < batch_size:
        return

    # Sample mini-batch
    # Calculate action-values using policy network
    # Calculate TD-targets using target network
    # Calculate loos
    # Make gradient descrent step and update policy network
    
    # Remove the next line once you have finished coding the function
    print('YOU MUST FINISH THE \'optimize_model()\' FUNCTION BEFORE THIS WILL WORK!')

    
# print('Optimizing model')
optimize_model()

YOU MUST FINISH THE 'optimize_model()' FUNCTION BEFORE THIS WILL WORK!


# Step 11: Periodically update the Fixed Q-value target

In [36]:
global steps_done
steps_done = 3

# Update the target network, copying all weights and biases from the policy network
if steps_done % target_update == 0:
    print("updating network")
    target_net.load_state_dict(policy_net.state_dict())

# Step 12: Evaluate Agent performance

One the agent has been trained (which might take some time) we need to evaluate the performance of the agent. We want to calculate the episode completion rate (the percentage of times the agent successfully reaches the goal), the average reward, and the average number of steps taken per episode.


In [37]:
# evaluation loop
finishCounter = 0.0
totalSteps = 0.0
totalReward = 0.0

steps_done = 1000000
stop_epsilon = 0.0
evalEpisodes = 2

for e in range(evalEpisodes):
    # Initialize the environment and state
    currentObs, _ = env.reset()
    currentState = preprocess(currentObs)
   
    # the main RL loop
    for i in range(0, env.max_steps):
        # Select and perform an action
        action = select_action(currentState)
        a = action.item()

        # take action 'a', receive reward 'reward', and observe next state 'obs'
        # 'done' indicate if the termination state was reached
        obs, reward, done, truncated, info = env.step(a)
        
        if (done or truncated):
            # Observe new state
            nextState = None
        else:
            nextState = preprocess(obs)

        if (done or truncated):
            totalReward += reward
            totalSteps += env.step_count
            if (done):
                print('Finished evaluation episode %d with reward %f,  %d steps, reaching goal ' % (e, reward, env.step_count))
                finishCounter += 1
            if (truncated):
                print('Failed evaluation episode %d with reward %f, %d steps' % (e,reward, env.step_count))
            break
        
        # Move to the next state
        currentState = nextState

# Print a summary of the evaluation results
print('Completion rate %.2f with average reward %0.4f and average steps %0.2f' % (finishCounter/evalEpisodes, totalReward/evalEpisodes,  totalSteps/evalEpisodes))

Failed evaluation episode 0 with reward 0.000000, 256 steps
Failed evaluation episode 1 with reward 0.000000, 256 steps
Completion rate 0.00 with average reward 0.0000 and average steps 256.00


# Assignment


## Task 1 - Deep Q-Networks

Implement an DQN agent that uses the partial observation (representing the objects in the tiles around the agent) as the state of the agent. The agent must also implement $\epsilon$-greedy exploration where the $\epsilon$-value must decay as training progresses. 

1. Train a policy for the agent for a maximum of 3000 episodes using DQN. Plot the reward as a function of the training steps using Tensorboard
2. Evaluate the performance of the trained policy by calculating the episode completion rate (the percentage of episode that the agent reached the goal), the average number of steps and the average reward over 1000 evaluation episodes.
3. Perform hyper-parameter tuning to find optimal training values that will result in the highest average reward and completion rate.

You have to make decisions regarding the following:

1. The neural network architecture
    - The number of neurons in the first hidden layer
    - The number of neurons in the second hidden layer
    - The non-linear activation function (although I recommend just using ReLU)
2. The training hyper-parameters
    - The learning rate $\alpha$
    - The discount factor $\gamma$
    - The interval (number of training steps) between fixed Q-value target updates
    - The size of the mini-batches
3. The exploration hyper-parameters
    - The maximum $\epsilon_{\textrm{max}}$ and miminum $\epsilon_{\textrm{mix}}$ values for the $\epsilon$-greedy threshold $\epsilon_{\textrm{threshold}}$ 
    - The epsilon decay rate $\epsilon_{\textrm{decay}}$
    
# Advanced Assignment (Optional)

## Task 2 - Deep Q-Networks with RGB images as observation

Implement an DQN agent that uses an 56x56 pixel RGB image (representing the top-down view of the agent) as the state of the agent. You will have to create a new neural network architecture for the policy network and target network.

1. Train a policy for the agent for a maximum of 3000 episodes using DQN. Plot the reward as a function of the training steps using Tensorboard
2. Evaluate the performance of the trained policy by calculating the episode completion rate (the percentage of episode that the agent reached the goal), the average number of steps and the average reward over 1000 evaluation episodes.
3. Perform hyper-parameter tuning to find optimal training values that will result in the highest average reward and completion rate.

You have to make decisions regarding the following:

1. The neural network architecture
    - The architecture of the feature learning layers
        - The number of Convolutional layers
        - The kernel size and stride of each CNN layers
    - The architecture of the function approximator layers
        - The number of hidden, fully-connected, feed-forward layers
        - The number of neurons in the each hidden layer
        - The non-linear activation function of each layer (although I recommend just using ReLU)
2. The training hyper-parameters
    - The learning rate $\alpha$
    - The discount factor $\gamma$
    - The interval (number of training steps) between fixed Q-value target updates
    - The size of the mini-batches
3. The exploration hyper-parameters
    - The maximum $\epsilon_{\textrm{max}}$ and miminum $\epsilon_{\textrm{mix}}$ values for the $\epsilon$-greedy threshold $\epsilon_{\textrm{threshold}}$ 
    - The epsilon decay rate $\epsilon_{\textrm{decay}}$

### Using RGB images as observations

The minigrid environment can be configure to provided a **56x56** pixel RGB image as environment observation (as shown below). This results in a more challenging task as the Neural Network function approximator not only needs to learn a policy, but only needs to learn features from the RGB image. The preprocessing of the RGB images used in this part of the assignment is taken from Thomas Simonini's _Deep Q Learning with Doom_ tutorial [https://simoninithomas.github.io/Deep_reinforcement_learning_Course]

![56x56 pixel RGB image](figures/observation_rot_enlarged.png)

We first need to configure the environment to give RGB images as observation, instead of the tile of object IDs as in the previous assignments:

In [39]:
env = gym.make('MiniGrid-Empty-8x8-v0')
env = RGBImgPartialObsWrapper(env)
env = ImgObsWrapper(env) # Get rid of the 'mission' field

### Preprocess RGB images

Preprocessing is an important step, <b>because we want to reduce the complexity of our states to reduce the computation time needed for training.</b>
<br><br>
Our steps:
- Grayscale each of our frames (because <b> color does not add important information </b>). 
- We normalize pixel values to be a value between [0,1]

This will result in the following grayscale image:
![56x56 pixel RGB image](figures/greyscale_observation_rot_enlarged.png)

In [40]:
### RGB Image parameters
screen_height = 56
screen_width = 56

# Compute grayscale version of RGB frame
def grayscale(frame):
    # Grayscale frame
    return np.mean(frame,-1)

# Normalize Pixel Values
def normalize_frame(frame, max_value):
    return frame/max_value

# Combine all the preprocessing fuctions into a single function
# This will give a frame of shape [screen_height, screen_width] having values between [0,1]
def preprocess_frame(frame):
    return normalize_frame(grayscale(frame), 255.0)

As explained in this really <a href="https://danieltakeshi.github.io/2016/11/25/frame-skipping-and-preprocessing-for-deep-q-networks-on-atari-2600-games/">  good article </a> we stack frames.

Stacking frames is really important because it helps us to **give have a sense of motion to our Neural Network.**

- First we preprocess frame
- Then we append the frame to the deque that automatically **removes the oldest frame**
- Finally we **build the stacked state**

This is how the stack works:
- For the first frame, we feed the stack 4 frames
- At each timestep, **we add the new frame to deque and then we stack them to form a new stacked frame**
- If we're done, **we create a new stack with 4 new frames (because we are in a new episode)**.

In [41]:
# Frame stack parameters
stack_size = 4

# Initialize deque with zero-images one array for each image
stacked_frames = deque([np.zeros((screen_height,screen_width), dtype=np.int32) for i in range(stack_size)], maxlen=stack_size) 

# We return both the stacked frames as a single tensor object, and the deque that contains the individual frames
# The boolean 'is_new_episode' is used to indicate that the input frame must be fed to the deque 'stack_size' times
def stack_frames(stacked_frames, state, is_new_episode):
    # Preprocess frame
    frame = preprocess_frame(state)
    
    if is_new_episode:
        # Clear our stacked_frames
        stacked_frames = deque([np.zeros((screen_height,screen_width), dtype=np.int32) for i in range(stack_size)], maxlen=stack_size)
        
        for i in range(stack_size):
            # Because we're in a new episode, copy the same frame 'stack_size' times
            stacked_frames.append(frame)
        
        # Stack the frames
        stacked_state = torch.from_numpy(np.stack(stacked_frames, axis=0)).float().unsqueeze(0)
        
    else:
        # Append frame to deque, automatically removes the oldest frame
        stacked_frames.append(frame)

        # Build the stacked state (first dimension specifies different frames)
        stacked_state = torch.from_numpy(np.stack(stacked_frames, axis=0)).float().unsqueeze(0)
    
    return stacked_state, stacked_frames

Below is an example of the initial observation being stacked 4 times

In [42]:
currentObs, info = env.reset()
currentState, stacked_frames = stack_frames(stacked_frames, currentObs, True)

print(currentState)
print(currentState.shape)

tensor([[[[0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          ...,
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980],
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980],
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980]],

         [[0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          ...,
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980],
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980],
          [0.5725, 0.5725, 0.5725,  ..., 0.2980, 0.2980, 0.2980]],

         [[0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0.5725, 0.5725, 0.5725],
          [0.5725, 0.5725, 0.5725,  ..., 0

# Step 5: Define and create our Convolution Neural Network model

This is our Deep Q-learning model for using images as imput:

![Title](figures/convnet.png)

- We preprocess the RGB image and create a stacked frame
- We take the stacked frame as input state
- We pass it through CNN-based feature learning layers, and the through fully-connected, feed-forward layers
- Each CNN layer has a 2D-convolutional layer, a 2D-Batch Normalization layer, and a ReLU non-linear activation function
    - A 2D-convolutional layer has the 'number of input channels', the 'number of output channels', the 'kernel size', and the 'stride' as parameters
    - The 2D-Batch Normalization layer has the 'number of features' as parameter. This should be the number of output channels of the preceding 2D-convolutional layer
- Each fully-connected, feed-forward layer has a ReLU non-linear activation function
- It outputs a Q value for each actions


The 2D-Batch Normalization layer applies Batch Normalization over a 4D input (a mini-batch of 2D inputs with additional channel dimension) as described in the paper `Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift <https://arxiv.org/abs/1502.03167>

In [43]:
### Neural network model definition
class CNN_DQN(nn.Module):

    def __init__(self, height, width, numActions, hiddenLayerSize=(512,)): 
        super(CNN_DQN, self).__init__()
     
        # h,w = 56, kernel_size = 5, stride = 2
        # input channels = 4 (one for each stacked frame)
        # output channels = 16
        
        # Input size is 4x56x56
        # Output size is 16x28x28
        self.conv1 = nn.Conv2d(4, 16, kernel_size=5, stride=2)
        self.bn1 = nn.BatchNorm2d(16)
        
        # Input is 16x26x26
        # Output is 32x14x14
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, stride=2)
        self.bn2 = nn.BatchNorm2d(32)
        
        # Input is 32x11x11
        # Output is 32x4x4
        self.conv3 = nn.Conv2d(32, 32, kernel_size=5, stride=2)
        self.bn3 = nn.BatchNorm2d(32)
        # Flattened size is 32x4x4 = 512

        # Number of Linear input connections depends on output of conv2d layers
        # and therefore the input image size, so compute it.
        def conv2d_size_out(size, kernel_size = 5, stride = 2):
            return (size - (kernel_size - 1) - 1) // stride  + 1
        convw = conv2d_size_out(conv2d_size_out(conv2d_size_out(width)))
        convh = conv2d_size_out(conv2d_size_out(conv2d_size_out(height)))
        linear_input_size = convw * convh * 32
        
        self.head = nn.Linear(linear_input_size, hiddenLayerSize[0])
        self.fc1 = nn.Linear(hiddenLayerSize[0], numActions)
        
    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = x.to(device)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x))) 

        x = F.relu(self.head(x.view(x.size(0), -1)))
        x = self.fc1(x)
        return x

    
# Instantiate the policy network and the target network
hiddenLayerSize = (128,)
policy_net = CNN_DQN(screen_height, screen_width, numActions, hiddenLayerSize)
target_net = CNN_DQN(screen_height, screen_width, numActions, hiddenLayerSize)

# Copy the weights of the policy network to the target network
target_net.load_state_dict(policy_net.state_dict())

# We don't want to update the parameters of the target network so we set it to evaluation mode
target_net.eval()

CNN_DQN(
  (conv1): Conv2d(4, 16, kernel_size=(5, 5), stride=(2, 2))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(2, 2))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 32, kernel_size=(5, 5), stride=(2, 2))
  (bn3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (head): Linear(in_features=512, out_features=128, bias=True)
  (fc1): Linear(in_features=128, out_features=3, bias=True)
)

# Instructor

Prof Herman Engelbrecht is the instructor for this practical assignment. He can be contacted via email [hebrecht@sun.ac.za](mailto:hebrecht@sun.ac.za)